In [10]:
from app.src.core_etl import get_html_content_metadata, refine_metadata_with_llm, run_pipeline
import asyncio

# `extract_html_data` ahora incluye un fallback que usa Playwright si es necesario

In [11]:
TEST_URL = "https://www.apa.org/topics/psychotherapy/psicoterapia"

# Ejecutamos la extracción principal (usa requests/cloudscraper y Playwright como fallback)
"""result = await get_html_content_metadata(TEST_URL)

# Mostrar un resumen rápido
print('METADATA_SEO:', result.get('metadata_seo'))
print('TEXTO_LIMPIO length:', len(result.get('texto_limpio', '')))
print('TEXTO_LIMPIO (primeros 1000 chars):')
print(result.get('content', ''))"""
# await run_pipeline(TEST_URL)

"result = await get_html_content_metadata(TEST_URL)\n\n# Mostrar un resumen rápido\nprint('METADATA_SEO:', result.get('metadata_seo'))\nprint('TEXTO_LIMPIO length:', len(result.get('texto_limpio', '')))\nprint('TEXTO_LIMPIO (primeros 1000 chars):')\nprint(result.get('content', ''))"

In [12]:
import asyncio
from app.src.core_etl import run_pipeline

async def batch_processor(urls):
    semaphore = asyncio.Semaphore(5)
    
    async def process_with_limit(url):
        async with semaphore:
            print(f"[*] Iniciando procesamiento de: {url}")
            await run_pipeline(url)

    tasks = [process_with_limit(url) for url in urls]
    await asyncio.gather(*tasks)

urls_de_prueba = [
        "https://medlineplus.gov/spanish/recetas/licuado-de-calabaza/",
        "https://retinia.mx/agudeza-visual-av/",
        "https://www.cdc.gov/spanish/niosh/topics/tiro.html",
        "http://familiasysexualidades.inmujeres.gob.mx/pdf/10.pdf",
        "https://medlineplus.gov/spanish/druginfo/meds/a622022-es.html"
        
    ]
await batch_processor(urls=urls_de_prueba)

[*] Iniciando procesamiento de: https://medlineplus.gov/spanish/recetas/licuado-de-calabaza/
[*] Descargando HTML de: https://medlineplus.gov/spanish/recetas/licuado-de-calabaza/
[*] Iniciando procesamiento de: https://retinia.mx/agudeza-visual-av/
[*] Descargando HTML de: https://retinia.mx/agudeza-visual-av/
[*] Iniciando procesamiento de: https://www.cdc.gov/spanish/niosh/topics/tiro.html
[*] Descargando HTML de: https://www.cdc.gov/spanish/niosh/topics/tiro.html
[-] Error al descargar https://www.cdc.gov/spanish/niosh/topics/tiro.html: 404 Client Error: Not Found for url: https://www.cdc.gov/spanish/niosh/topics/tiro.html
[*] Iniciando procesamiento de: http://familiasysexualidades.inmujeres.gob.mx/pdf/10.pdf
[*] Descargando HTML de: http://familiasysexualidades.inmujeres.gob.mx/pdf/10.pdf
[-] Error al descargar http://familiasysexualidades.inmujeres.gob.mx/pdf/10.pdf: HTTPConnectionPool(host='familiasysexualidades.inmujeres.gob.mx', port=80): Max retries exceeded with url: /pdf/10

PermissionError: No fue posible hacer web-scrapping de: http://familiasysexualidades.inmujeres.gob.mx/pdf/10.pdf 
 Not enough content retrieved

[*] Contenido sin cambios, saltando.

=== PRUEBA DE PIPELINE EXITOSA ===
[*] Contenido sin cambios, saltando.

=== PRUEBA DE PIPELINE EXITOSA ===
[*] Contenido sin cambios, saltando.

=== PRUEBA DE PIPELINE EXITOSA ===
[!] Error en el pipeline: No fue posible hacer web-scrapping de: https://www.cdc.gov/spanish/niosh/topics/tiro.html 
 page not found


In [9]:
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
import os
import dotenv
dotenv.load_dotenv()
# Configurar el cliente de búsqueda
endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
key = os.getenv("AZURE_SEARCH_KEY")
index_name = os.getenv("AZURE_SEARCH_INDEX_NAME")

client = SearchClient(endpoint, index_name, AzureKeyCredential(key))

# Realizar una búsqueda de conteo
results = client.search(search_text="*", include_total_count=True)
print(f"Total de documentos en el índice: {results.get_count()}")

# Ver un ejemplo de cómo guardó Azure tu documento
resultados = client.search(
    search_text="obesidad",
    filter="type eq 'summary'",
    top=3# 'eq' significa equal (igual a)
)

for doc in resultados:
    print(doc['title'], doc['type'], doc['url'])

Total de documentos en el índice: 1813
La Obesidad en el Menor de Edad | Sitio Web "Acercando el IMSS al Ciudadano" summary http://www.imss.gob.mx/salud-en-linea/obesidad-menoredad
Unknown summary http://www.imss.gob.mx/salud-en-linea/sobrepeso-obesidad
Las discapacidades y la obesidad  | Las discapacidades y la salud | NCBDDD | CDC summary https://www.cdc.gov/ncbddd/spanish/disabilityandhealth/obesity.html
